In [0]:
from datetime import datetime
from pyspark.sql.functions import col, when, lower, trim, current_timestamp, to_timestamp, lit, regexp_replace,\
to_date, input_file_name
from pyspark.sql.types import DecimalType, StringType, IntegerType, TimestampType

dbutils.widgets.text("p_source_folder","bronze")
dbutils.widgets.text("p_target_table","")
dbutils.widgets.text("p_date_path","")
dbutils.widgets.text("p_file_name","")
dbutils.widgets.text("p_primary_key","")

source_folder = dbutils.widgets.get("p_source_folder")
target_table = dbutils.widgets.get("p_target_table")
file_name = dbutils.widgets.get("p_file_name")
date_path = dbutils.widgets.get("p_date_path")
primary_key = dbutils.widgets.get("p_primary_key")


if not target_table or not date_path or not file_name:
    raise Exception(f"Required parameters missing. Table: {target_table}, Date: {date_path}, {file_name}")

In [0]:
storage_account = "adlsgen2dreamprod" 
sink_container = "silver"

In [0]:
bronze_path = f"abfss://{source_folder}@{storage_account}.dfs.core.windows.net/dream_app/{target_table}/{date_path}/{file_name}_*.csv"

df_raw = spark.read.format("csv")\
    .options(header="true", inferschema="true", multiline="true", escape='"')\
    .load(bronze_path)\
    .withColumn("_source_file", col("_metadata.file_path"))

clean_expressions = []

for original_name in df_raw.columns:
    clean_name = original_name.lower().strip().replace(" ", "_").replace(".", "_").replace("-", "_").replace("(","")\
                              .replace(")", "").replace("/", "_").replace("#", "")
    target_col = col(original_name)

    if any(word in clean_name for word in ["name", "first", "last"]):
        exp = regexp_replace(target_col,r"[^a-zA-Z\s]", "")
        exp = trim(lower(exp)).cast(StringType())
        exp = when((exp.isNull()) | (exp == ""), "N/A").otherwise(exp)
        
    elif any(word in clean_name for word in ["price", "amount", "fare", "total", "cost"]):
        exp = regexp_replace(target_col, r"[^0-9\.]", "").cast(DecimalType(18,2))
        exp = when(exp.isNull(), lit(0.00)).otherwise(exp)

    elif any(word in clean_name for word in ["date", "timestamp", "time"]):
        exp = to_timestamp(target_col)
        exp = when(exp.isNull(), to_timestamp(lit("1990-01-01 00:00:00"))).otherwise(exp)

    elif any(word in clean_name for word in ["email"]):
        exp = lower(regexp_replace(target_col, r"[^a-zA-Z0-9@\.\-\_]", ""))
        exp = when((exp.isNull()) | (exp == ""), "N/A").otherwise(exp)
    
    elif "id" in clean_name :
        exp = target_col.cast(IntegerType())
        exp = when(exp.isNull(), 0).otherwise(exp)
    
    else :
        col_type = [t for n, t in df_raw.dtypes if n == original_name][0]
        if col_type == "string":
            exp = trim(lower(target_col))
            exp = when((exp.isNull()) | (exp == ""), "N/A").otherwise(exp)
        else:
            exp = target_col
    
    clean_expressions.append(exp.alias(clean_name))

df_silver = df_raw.select(*clean_expressions)\
    .withColumn("_processed_at", current_timestamp())

clean_pk = None
if primary_key:
    clean_pk = primary_key.lower().strip().replace(" ", "_").replace(".", "_").replace("-", "_") \
                          .replace("(", "").replace(")", "").replace("/", "_").replace("#", "")
    if clean_pk in df_silver.columns:
        df_silver = df_silver.dropDuplicates([clean_pk])
    else:
        df_silver = df_silver.dropDuplicates()
else:
    df_silver = df_silver.dropDuplicates()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS adb_dream_analytics_prod.silver MANAGED LOCATION 'abfss://silver@{storage_account}.dfs.core.windows.net/dream_app/'")

gold_table_name = f"adb_dream_analytics_prod.silver.{target_table}"

df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(gold_table_name)

if clean_pk:
    spark.sql(f"OPTIMIZE {gold_table_name} ZORDER BY (`{clean_pk}`)")
else:
    spark.sql(f"OPTIMIZE {gold_table_name}")

print(f"Successfully processed {target_table} to Silver Delta.")
    
    

In [0]:
%sql
SELECT * FROM adb_dream_analytics_prod.silver.trips
LIMIT 10;